# Kayıp Fonksiyonları

Bu alıştırmada, Kayıp fonksiyonlarının `LinearRegression` modeli üzerindeki etkilerini karşılaştıracaksınız.

👇 Bu zorluk için kullanmak üzere bir CSV dosyası indirelim ve onu bir DataFrame'e dönüştürelim

In [1]:
import pandas as pd

data = pd.read_csv("https://d32aokrjazspmn.cloudfront.net/materials/loss_functions_dataset.csv")
data.sample(5)

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area,Average Temperature
364,0.71,710.5,269.5,220.5,3.5,0.25,13.620
13,0.82,612.5,318.5,147.0,7.0,0.00,19.435
694,0.76,661.5,416.5,122.5,7.0,0.40,40.225
638,0.82,612.5,318.5,147.0,7.0,0.40,29.620
635,0.86,588.0,294.0,147.0,7.0,0.40,31.720


🎯 Göreviniz, tasarımına göre bir seranın içindeki ortalama sıcaklığı tahmin etmektir. Sıcaklık tahminleriniz, her bir bitki için iklim ihtiyaçlarına göre uygun sera tasarımını seçmenize yardımcı olacaktır.

🌿 Bitkilerin küçük sıcaklık değişimlerini kaldırabildiğini, ancak sıcaklık değişimleri arttıkça katlanarak daha duyarlı hale geldiğini biliyorsunuz.

## 1. Teori

❓ Teorik olarak, bitkileri öldürme riskini sınırlamak için modelinizi hangi Kayıp fonksiyonu üzerinde eğitirsiniz?

<details>
<summary> 🆘 Cevap </summary>
    
Teorik olarak, Ortalama Kare Hata (MSE) Kayıp fonksiyonunu kullanırsınız. Bu, aykırı tahminleri cezalandırır ve modelinizin büyük hatalar yapmasını engeller. Bu, daha küçük sıcaklık değişimleri ve bitkiler için daha düşük risk sağlayacaktır.

</details>

## 2. Uygulama

### 2.1 Ön İşleme

❓ Özellikleri standartlaştırın

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pandas as pd

# X / y
X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

# split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# standardize
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

X_train_scaled.head()

,Relative Compactness,Surface Area,Wall Area,Roof Area,Overall Height,Glazing Area
60,0.553671,-0.696222,-0.007372,-0.679048,1.016421,-1.010300
618,-1.155118,1.250664,0.558439,0.957063,-0.983844,1.227790
346,0.933402,-0.974349,-0.573184,-0.679048,1.016421,0.108745
294,1.313133,-1.252476,-0.007372,-1.224418,1.016421,0.108745
231,-0.965252,0.972537,-0.007372,0.957063,-0.983844,-1.010300


### 2.2 Modelleme

Bu bölümde, farklı Kayıp fonksiyonları üzerinde optimize edilmiş modelleri değerlendirerek teoriyi doğrulayacaksınız.

### En Küçük Kareler (MSE) Kaybı

❓ **En Küçük Kareler Kaybı** (MSE) üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

In [4]:
import numpy as np
import pandas as pd
from time import perf_counter

from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor

# X / y
X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

# Pipeline: scaler + SGD (MSE)
pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(
        loss="squared_error",   # MSE loss
        penalty=None,           # saf SGD 
        max_iter=5000,
        tol=1e-4,
        random_state=42
    ))
])

cv = KFold(n_splits=10, shuffle=True, random_state=42)

t0 = perf_counter()
scores = cross_val_score(
    pipe, X, y,
    cv=cv,
    scoring="neg_mean_squared_error"  # sklearn "büyük daha iyi" diye negatif döner
)
elapsed = perf_counter() - t0

mse_scores = -scores  # negatifi pozitife çeviriyoruz

print("Fold MSE:", mse_scores)
print("Mean MSE:", mse_scores.mean())
print("Std  MSE:", mse_scores.std())
print("Time (s):", elapsed)

Fold MSE: [ 9.68351751  7.57120824  8.71547489  7.37607585  7.53440242 12.31426644
  7.48761491  8.45714737  9.35175331  7.53106718]
Mean MSE: 8.602252814306809
Std  MSE: 1.471071989097395
Time (s): 0.09855662999996184


❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru ve bunu `r2` değişkeninde kaydedin
- Tüm katlarınızın °C cinsinden en büyük tek tahmin hatasını hesaplayın ve `max_error_celsius` değişkeninde kaydedin

(İpucu: `max_error` sklearn'de kabul edilen bir puanlama metriğidir)

In [7]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor

X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

pipe = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(
        loss="squared_error",
        penalty=None,
        max_iter=5000,
        tol=1e-4,
        random_state=42
    ))
])

cv = KFold(n_splits=10, shuffle=True, random_state=42)

# 1) Ortalama CV R2
r2_scores = cross_val_score(pipe, X, y, cv=cv, scoring="r2")
r2 = r2_scores.mean()

# 2) Fold'lar bazında "en büyük tek hata" (negatif döner)
neg_max_err_scores = cross_val_score(pipe, X, y, cv=cv, scoring="neg_max_error")

# Her fold için max error (°C) = -neg_max_error
max_err_per_fold = -neg_max_err_scores

# Tüm fold'lar içinde görülen en büyük tek hata (°C)
max_error_celsius = max_err_per_fold.max()

r2, max_error_celsius

(0.908334303194626, 9.110526076101245)

### Ortalama Mutlak Hata (MAE) Kaybı

Peki modelimizi MAE üzerinde optimize edersek ne olur?

❓ **MAE** Kaybı üzerinde **Stokastik Gradyan İnişi** (SGD) ile optimize edilmiş bir Doğrusal Regresyon modelini **10-Katlı Çapraz doğrula**

<details>
<summary>💡 İpuçları</summary>

- MAE kaybı `SGDRegressor`'da doğrudan belirtilemez. Doğru parametreleri ayarlayarak tasarlanması gerekir

</details>

In [8]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor

X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

pipe_mae = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(
        loss="epsilon_insensitive",  # L1 benzeri
        epsilon=0.0,                 # => |y - yhat| (MAE)
        penalty=None,
        max_iter=5000,
        tol=1e-4,
        random_state=42
    ))
])

cv = KFold(n_splits=10, shuffle=True, random_state=42)

# İstersen MAE ile raporla (performans metriği)
neg_mae_scores = cross_val_score(pipe_mae, X, y, cv=cv, scoring="neg_mean_absolute_error")
mae_cv = (-neg_mae_scores).mean()

# Ek: R2'yi de görmek istersen
r2_scores = cross_val_score(pipe_mae, X, y, cv=cv, scoring="r2")
r2_cv = r2_scores.mean()

mae_cv, r2_cv

(2.07269217901276, 0.903997700642664)

❓ Hesaplayın:
- Ortalama çapraz doğrulanmış R2 skoru, bunu `r2_mae`'de saklayın
- Tüm katlarınızın en büyük tek tahmin hatasını, bunu `max_error_mae`'de saklayın

In [9]:
import numpy as np
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor

X = data.drop(columns=["Average Temperature"])
y = data["Average Temperature"]

pipe_mae = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("sgd", SGDRegressor(
        loss="epsilon_insensitive",
        epsilon=0.0,      # => MAE (L1)
        penalty=None,
        max_iter=5000,
        tol=1e-4,
        random_state=42
    ))
])

cv = KFold(n_splits=10, shuffle=True, random_state=42)

# Ortalama CV R2
r2_scores_mae = cross_val_score(pipe_mae, X, y, cv=cv, scoring="r2")
r2_mae = r2_scores_mae.mean()

# Fold bazında max error (negatif döner)
neg_max_err_scores_mae = cross_val_score(pipe_mae, X, y, cv=cv, scoring="neg_max_error")
max_err_per_fold_mae = -neg_max_err_scores_mae

# Tüm fold'lar içindeki en büyük tek hata (°C)
max_error_mae = max_err_per_fold_mae.max()

r2_mae, max_error_mae

(0.903997700642664, 9.75235176571848)

## 3. Sonuç

❓ Değerlendirdiğiniz modellerden hangisi göreviniz için en uygun görünüyor?

<details>
<summary> 🆘Cevap </summary>
    
İki model arasında ortalama çapraz doğrulanmış r2 skorları yaklaşık olarak benzer olmasına rağmen, MAE üzerinde optimize edilen modelin zaman zaman daha büyük hatalar yapma şansı daha fazladır, bu da bitkileri öldürme riskini artırır!
    
</details>


> MSE ile optimize edilen model:

CV mean R² = 0.9083

Worst single error = 9.11°C

>MAE ile optimize edilen model:

CV mean R² = 0.9040

Worst single error = 9.75°C

Bu yüzden MSE üzerinde optimize edilen SGDRegressor bu görev için daha uygun görünüyor

# 🏁 Kodunuzu kontrol edin ve notebook'unuzu gönderin

In [10]:
from nbresult import ChallengeResult

result = ChallengeResult(
    'loss_functions',
    r2 = r2,
    r2_mae = r2_mae,
    max_error = max_error_celsius,
    max_error_mae = max_error_mae
)

result.write()
print(result.check())


============================= test session starts ==============================
platform linux -- Python 3.12.9, pytest-8.3.4, pluggy-1.5.0 -- /home/mcelik/.pyenv/versions/3.12.9/envs/workintech/bin/python
cachedir: .pytest_cache
rootdir: /home/mcelik/code/S16D4-S-loss-functions/tests
plugins: typeguard-4.4.2, anyio-4.8.0
collecting ... collected 3 items

test_loss_functions.py::TestLossFunctions::test_max_error_order PASSED   [ 33%]
test_loss_functions.py::TestLossFunctions::test_r2 PASSED                [ 66%]
test_loss_functions.py::TestLossFunctions::test_r2_mae PASSED            [100%]

============================== 3 passed in 0.26s ===============================


💯 You can commit your code:

git add tests/loss_functions.pickle

git commit -m 'Completed loss_functions step'

git push origin master

